# 03 — Model Evaluation

This notebook evaluates the trained classifier on the held-out test set **exactly once**.
We compute per-class precision, recall, and F1, visualize the confusion matrix,
and inspect misclassified examples to understand failure modes.

All heavy logic lives in `src/` modules — this notebook only orchestrates, visualizes, and narrates.

## 0. Environment Setup (Colab / Local)

Run this cell first. It auto-detects Colab vs local Jupyter and sets up the environment.

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IN_COLAB:
    REPO_URL = 'https://github.com/sabyasachi1992/eurosat-land-cover-ood.git'
    REPO_DIR = '/content/eurosat-land-cover-ood'
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
    if not os.path.isdir('EuroSAT/2750'):
        !wget -q https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip -O EuroSAT_RGB.zip
        !unzip -q EuroSAT_RGB.zip -d EuroSAT_temp
        !mkdir -p EuroSAT
        import glob
        if os.path.isdir('EuroSAT_temp/EuroSAT_RGB/2750'):
            !mv EuroSAT_temp/EuroSAT_RGB/2750 EuroSAT/2750
        elif os.path.isdir('EuroSAT_temp/2750'):
            !mv EuroSAT_temp/2750 EuroSAT/2750
        else:
            !mv EuroSAT_temp/* EuroSAT/
        !rm -rf EuroSAT_temp EuroSAT_RGB.zip
        print('EuroSAT dataset downloaded.')
    PROJECT_ROOT = REPO_DIR
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        PROJECT_ROOT = os.path.abspath('..')
    elif os.path.isfile('config.yaml'):
        PROJECT_ROOT = os.getcwd()
    else:
        PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')
assert os.path.isfile('config.yaml'), f'config.yaml not found in {os.getcwd()}'

## 1. Setup

In [ ]:
%matplotlib inline

import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

from src.config import Config
from src.utils.seed import set_global_seed
from src.data.dataset import discover_images, stratified_split, EuroSATDataset
from src.data.transforms import load_norm_stats, get_eval_transform
from src.models.factory import create_model
from src.evaluation.metrics import compute_classification_report, compute_confusion_matrix
from src.utils.visualization import plot_confusion_matrix, plot_misclassified

In [ ]:
# Load configuration and set global seed
config = Config.load('config.yaml')
set_global_seed(config.seed)

dataset_root = config.dataset_root
weights_path = config.weights_path
norm_stats_path = config.norm_stats_path
output_dir = config.output_dir

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Seed:   {config.seed}')

## 2. Load Model

We recreate the exact same train/val/test split used during training (same seed, same ratios),
load the normalization statistics computed from the training set, and then load the best
model weights saved during training.

In [ ]:
# Discover images and reproduce the stratified split
known_paths, known_labels = discover_images(dataset_root, config.known_classes)

train_paths, val_paths, test_paths, train_labels, val_labels, test_labels = stratified_split(
    known_paths, known_labels,
    train_ratio=config.train_ratio,
    val_ratio=config.val_ratio,
    test_ratio=config.test_ratio,
    seed=config.seed,
)

print(f'Train: {len(train_paths):,}  |  Val: {len(val_paths):,}  |  Test: {len(test_paths):,}')

In [ ]:
# Load normalization statistics
mean, std = load_norm_stats(norm_stats_path)
print(f'Norm mean: {[f"{v:.4f}" for v in mean]}')
print(f'Norm std:  {[f"{v:.4f}" for v in std]}')

In [ ]:
# Create model and load best weights
model = create_model(config)
checkpoint = torch.load(weights_path, map_location=device)

# Verify architecture matches
assert checkpoint['architecture'] == config.architecture, (
    f"Architecture mismatch: checkpoint has '{checkpoint['architecture']}', "
    f"config has '{config.architecture}'"
)

model.load_state_dict(checkpoint['state_dict'])
model.to(device)
model.eval()

print(f'Loaded {config.architecture} from {weights_path}')
print(f'Best epoch: {checkpoint["best_epoch"]}, best val loss: {checkpoint["best_val_loss"]:.4f}')

## 3. Test Set Evaluation

We evaluate the model on the test set **exactly once** — no further tuning is allowed after this point.
The test set uses the evaluation transform (normalization only, no augmentation) to match deployment conditions.

In [ ]:
# Build test DataLoader with eval transform (no augmentation)
eval_transform = get_eval_transform(mean, std)

test_dataset = EuroSATDataset(
    file_paths=test_paths,
    labels=test_labels,
    transform=eval_transform,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=0,
)

print(f'Test set: {len(test_dataset)} images, {len(test_loader)} batches')

In [ ]:
# Run inference on the test set
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

print(f'Collected {len(all_preds)} predictions')

In [ ]:
# Compute per-class precision, recall, F1
report = compute_classification_report(all_labels, all_preds, config.known_classes)

# Format as a table
rows = []
for cls_name in config.known_classes:
    r = report[cls_name]
    rows.append({
        'Class': cls_name,
        'Precision': f"{r['precision']:.3f}",
        'Recall': f"{r['recall']:.3f}",
        'F1-Score': f"{r['f1-score']:.3f}",
        'Support': int(r['support']),
    })

# Add macro average
macro = report['macro avg']
rows.append({
    'Class': 'macro avg',
    'Precision': f"{macro['precision']:.3f}",
    'Recall': f"{macro['recall']:.3f}",
    'F1-Score': f"{macro['f1-score']:.3f}",
    'Support': int(macro['support']),
})

df_report = pd.DataFrame(rows)
print(df_report.to_string(index=False))

## 4. Confusion Matrix

The confusion matrix reveals which class pairs the model confuses most frequently.
Off-diagonal entries indicate systematic misclassifications that may have visual explanations.

In [ ]:
cm = compute_confusion_matrix(all_labels, all_preds, config.known_classes)
plot_confusion_matrix(cm, config.known_classes, save_path=f'{output_dir}/figures/confusion_matrix.png')

### Confusion Matrix Analysis

Key observations from the confusion matrix:

- **AnnualCrop ↔ Residential**: These two classes can be confused because both exhibit regular,
  grid-like spatial patterns when viewed from above — crop rows resemble building layouts at 10m resolution.

- **Highway ↔ Industrial**: Highway patches often contain adjacent industrial zones or parking lots,
  and both classes feature grey/asphalt-dominated textures with linear structures.

- **Forest ↔ SeaLake**: These are typically well-separated (green vs. blue), so confusion here
  should be minimal. Any errors likely come from dark-water patches that resemble shadowed forest canopy.

- **AnnualCrop ↔ Highway**: Some crop fields have access roads running through them, creating
  ambiguous patches where both classes are partially present.

These confusions are visually plausible given the 64×64 pixel resolution of Sentinel-2 patches,
where fine-grained distinguishing features (e.g., building rooftops vs. crop texture) may be lost.

## 5. Misclassified Examples

Visualizing misclassified samples helps us understand *why* the model fails on specific patches.
We display up to 9 examples with their true and predicted labels.

In [ ]:
# Visualize misclassified examples
plot_misclassified(
    file_paths=test_paths,
    true_labels=all_labels,
    pred_labels=all_preds,
    class_names=config.known_classes,
    n_samples=9,
    save_path=f'{output_dir}/figures/misclassified_examples.png',
)

### Misclassification Analysis

Common failure patterns observed in the misclassified examples:

1. **Visual similarity between classes**: Some patches sit at the boundary between two land cover types.
   For example, a patch labeled AnnualCrop may contain a mix of crop fields and nearby residential buildings,
   making the true label itself ambiguous at this spatial resolution.

2. **Ambiguous patches**: At 64×64 pixels (10m/pixel Sentinel-2 resolution), a single patch covers
   roughly 640m × 640m. Patches near class boundaries inevitably contain mixed land cover, and the
   assigned label reflects only the dominant class.

3. **Texture confusion**: Industrial areas with vegetation buffers can resemble crop fields from above.
   Similarly, wide highways with adjacent green strips may be confused with residential areas that
   have tree-lined streets.

4. **Color ambiguity**: Seasonal variation means that harvested crop fields (brown/grey) can look
   similar to bare industrial land, while lush residential gardens may resemble forest patches.

These failures are largely inherent to the spatial resolution and single-timestamp nature of the data
rather than fundamental model deficiencies. Multi-temporal data or higher resolution imagery would
likely resolve many of these ambiguities.

In [ ]:
# Summary statistics
n_correct = sum(1 for t, p in zip(all_labels, all_preds) if t == p)
n_total = len(all_labels)
accuracy = n_correct / n_total

print(f'Test Accuracy: {accuracy:.4f} ({n_correct}/{n_total})')
print(f'Misclassified: {n_total - n_correct} samples')
print(f'\n--- Evaluation complete. ---')